# XAI-Based Informative MRI Slice Selection

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/models/efficientnet_baseline.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Baseline EfficientNetB0 loaded successfully!")

Baseline EfficientNetB0 loaded successfully!


In [ ]:
import pandas as pd

META_DIR = "/content/drive/MyDrive/ASD_XAI_Project/metadata"

train_slices_df = pd.read_csv(f"{META_DIR}/train_slices.csv")
val_slices_df = pd.read_csv(f"{META_DIR}/val_slices.csv")
test_slices_df = pd.read_csv(f"{META_DIR}/test_slices.csv")

print("Train slices:", len(train_slices_df))
print("Validation slices:", len(val_slices_df))
print("Test slices:", len(test_slices_df))

Train slices: 4536
Validation slices: 1054
Test slices: 1036


## SHAP-Based Slice Importance

In [ ]:
!pip install -q shap

import shap
import numpy as np
import tensorflow as tf

IMG_SIZE = (224, 224)

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image

In [ ]:
# Build one representative reference slice per training subject

reference_records = []

for subject_id, group in train_slices_df.groupby("SUB_ID"):
    group = group.sort_values("SLICE_PATH").reset_index(drop=True)

    # Choose the middle available slice for this subject
    middle_index = len(group) // 2
    reference_records.append(group.iloc[middle_index])

reference_df = pd.DataFrame(reference_records).reset_index(drop=True)

reference_images = np.stack([
    load_image(path).numpy()
    for path in reference_df["SLICE_PATH"]
])

print("SHAP reference images:", reference_images.shape)

print("\nReference label distribution:")
print(reference_df["LABEL"].value_counts())

SHAP reference images: (26, 224, 224, 3)

Reference label distribution:
LABEL
ASD        13
Control    13
Name: count, dtype: int64


In [ ]:
explainer = shap.GradientExplainer(
    model,
    reference_images
)

print("SHAP GradientExplainer created successfully!")

SHAP GradientExplainer created successfully!


In [ ]:
import os

SHAP_BATCH_SIZE = 8
shap_results = []

for start in range(0, len(val_slices_df), SHAP_BATCH_SIZE):

    end = min(start + SHAP_BATCH_SIZE, len(val_slices_df))

    batch_df = val_slices_df.iloc[start:end]

    batch_images = np.stack([
        load_image(path).numpy()
        for path in batch_df["SLICE_PATH"]
    ])

    # Compute SHAP values
    shap_values = explainer.shap_values(
        batch_images,
        nsamples=50
    )

    # Handle single-output model
    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    # Remove final singleton output dimension if present
    if shap_values.ndim == 5:
        shap_values = np.squeeze(shap_values, axis=-1)

    # One SHAP score per MRI slice
    # Mean absolute contribution across all pixels/channels
    slice_scores = np.mean(
        np.abs(shap_values),
        axis=(1, 2, 3)
    )

    for i, (_, row) in enumerate(batch_df.iterrows()):
        shap_results.append({
            "SUB_ID": row["SUB_ID"],
            "SLICE_PATH": row["SLICE_PATH"],
            "LABEL": row["LABEL"],
            "SHAP_SCORE": float(slice_scores[i])
        })

    print(
        f"SHAP processed: {end}/{len(val_slices_df)} slices"
    )


shap_scores_df = pd.DataFrame(shap_results)

SHAP_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/results/"
    "shap_validation_scores.csv"
)

shap_scores_df.to_csv(
    SHAP_OUTPUT,
    index=False
)

print("\nSHAP analysis completed!")
print("Total slices scored:", len(shap_scores_df))
print("\nTop SHAP-ranked slices:")
print(
    shap_scores_df
    .sort_values("SHAP_SCORE", ascending=False)
    .head(10)
)

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(8, 224, 224, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(50, 224, 224, 3))']
  warnings.warn(msg)


SHAP processed: 8/1054 slices
SHAP processed: 16/1054 slices
SHAP processed: 24/1054 slices
SHAP processed: 32/1054 slices
SHAP processed: 40/1054 slices
SHAP processed: 48/1054 slices
SHAP processed: 56/1054 slices
SHAP processed: 64/1054 slices
SHAP processed: 72/1054 slices
SHAP processed: 80/1054 slices
SHAP processed: 88/1054 slices
SHAP processed: 96/1054 slices
SHAP processed: 104/1054 slices
SHAP processed: 112/1054 slices
SHAP processed: 120/1054 slices
SHAP processed: 128/1054 slices
SHAP processed: 136/1054 slices
SHAP processed: 144/1054 slices
SHAP processed: 152/1054 slices
SHAP processed: 160/1054 slices
SHAP processed: 168/1054 slices
SHAP processed: 176/1054 slices
SHAP processed: 184/1054 slices
SHAP processed: 192/1054 slices
SHAP processed: 200/1054 slices
SHAP processed: 208/1054 slices
SHAP processed: 216/1054 slices
SHAP processed: 224/1054 slices
SHAP processed: 232/1054 slices
SHAP processed: 240/1054 slices
SHAP processed: 248/1054 slices
SHAP processed: 256/1

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: input_layer
Received: inputs=['Tensor(shape=(6, 224, 224, 3))']
  warnings.warn(msg)


SHAP processed: 1054/1054 slices

SHAP analysis completed!
Total slices scored: 1054

Top SHAP-ranked slices:
     SUB_ID                                         SLICE_PATH    LABEL  \
832   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
812   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
811   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
29    51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
841   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
853   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
813   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
824   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
326   51460  /content/drive/MyDrive/ASD_XAI_Project/slices/...      ASD   
810   51476  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   

     SHAP_SCORE  
832    0.000357  
812    0.000341  
811    0.0

In [ ]:
import os
import pandas as pd

SHAP_OUTPUT = "/content/drive/MyDrive/ASD_XAI_Project/results/shap_validation_scores.csv"

print("File exists:", os.path.exists(SHAP_OUTPUT))

saved_shap = pd.read_csv(SHAP_OUTPUT)

print("Saved rows:", len(saved_shap))
print("Columns:", saved_shap.columns.tolist())

File exists: True
Saved rows: 1054
Columns: ['SUB_ID', 'SLICE_PATH', 'LABEL', 'SHAP_SCORE']


In [4]:
import os
import pandas as pd

SHAP_OUTPUT = "/content/drive/MyDrive/ASD_XAI_Project/results/shap_validation_scores.csv"

saved_shap = pd.read_csv(SHAP_OUTPUT)

print("File exists:", os.path.exists(SHAP_OUTPUT))
print("Saved rows:", len(saved_shap))
print("Columns:", saved_shap.columns.tolist())

print("\nFirst 5 rows:")
print(saved_shap.head())

File exists: True
Saved rows: 1054
Columns: ['SUB_ID', 'SLICE_PATH', 'LABEL', 'SHAP_SCORE']

First 5 rows:
   SUB_ID                                         SLICE_PATH    LABEL  \
0   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
1   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
2   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
3   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
4   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   

   SHAP_SCORE  
0    0.000295  
1    0.000283  
2    0.000245  
3    0.000214  
4    0.000264  


In [9]:
val_slices_df = saved_shap[
    ["SUB_ID", "SLICE_PATH", "LABEL"]
].copy()

print("Validation slices restored:", len(val_slices_df))
print(val_slices_df.head())

Validation slices restored: 1054
   SUB_ID                                         SLICE_PATH    LABEL
0   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
1   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
2   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
3   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control
4   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control


In [11]:
import os

PROJECT_DIR = "/content/drive/MyDrive/ASD_XAI_Project"

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if file.endswith((".keras", ".h5")):
            print(os.path.join(root, file))

/content/drive/MyDrive/ASD_XAI_Project/models/efficientnet_baseline.keras


In [12]:
import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/ASD_XAI_Project/models/efficientnet_baseline.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully!")
print(model.input_shape)
print(model.output_shape)

Model loaded successfully!
(None, 224, 224, 3)
(None, 1)


In [13]:
import tensorflow as tf

IMG_SIZE = 224

def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    return image

In [14]:
test_img = load_image(val_slices_df.iloc[0]["SLICE_PATH"])

print("Image shape:", test_img.shape)
print("Data type:", test_img.dtype)

Image shape: (224, 224, 3)
Data type: <dtype: 'float32'>


In [15]:
import numpy as np
import pandas as pd

PERM_BATCH_SIZE = 16
permutation_results = []

rng = np.random.default_rng(42)

for start in range(0, len(val_slices_df), PERM_BATCH_SIZE):

    end = min(start + PERM_BATCH_SIZE, len(val_slices_df))
    batch_df = val_slices_df.iloc[start:end]

    # Load original images
    batch_images = np.stack([
        load_image(path).numpy()
        for path in batch_df["SLICE_PATH"]
    ])

    # Original predictions
    original_preds = model.predict(
        batch_images,
        verbose=0
    ).reshape(-1)

    # Create permuted versions
    permuted_images = batch_images.copy()

    for i in range(len(permuted_images)):
        flat = permuted_images[i].reshape(-1).copy()
        rng.shuffle(flat)
        permuted_images[i] = flat.reshape(
            permuted_images[i].shape
        )

    # Predictions after permutation
    permuted_preds = model.predict(
        permuted_images,
        verbose=0
    ).reshape(-1)

    # Importance = change in prediction
    permutation_scores = np.abs(
        original_preds - permuted_preds
    )

    for i, (_, row) in enumerate(batch_df.iterrows()):

        permutation_results.append({
            "SUB_ID": row["SUB_ID"],
            "SLICE_PATH": row["SLICE_PATH"],
            "LABEL": row["LABEL"],
            "PERMUTATION_SCORE": float(
                permutation_scores[i]
            )
        })

    print(
        f"Permutation processed: "
        f"{end}/{len(val_slices_df)} slices"
    )


permutation_scores_df = pd.DataFrame(
    permutation_results
)

PERM_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/results/"
    "permutation_validation_scores.csv"
)

permutation_scores_df.to_csv(
    PERM_OUTPUT,
    index=False
)

print("\nPermutation analysis completed!")
print(
    "Total slices scored:",
    len(permutation_scores_df)
)

print("\nTop permutation-ranked slices:")

print(
    permutation_scores_df
    .sort_values(
        "PERMUTATION_SCORE",
        ascending=False
    )
    .head(10)
)

Permutation processed: 16/1054 slices
Permutation processed: 32/1054 slices
Permutation processed: 48/1054 slices
Permutation processed: 64/1054 slices
Permutation processed: 80/1054 slices
Permutation processed: 96/1054 slices
Permutation processed: 112/1054 slices
Permutation processed: 128/1054 slices
Permutation processed: 144/1054 slices
Permutation processed: 160/1054 slices
Permutation processed: 176/1054 slices
Permutation processed: 192/1054 slices
Permutation processed: 208/1054 slices
Permutation processed: 224/1054 slices
Permutation processed: 240/1054 slices
Permutation processed: 256/1054 slices
Permutation processed: 272/1054 slices
Permutation processed: 288/1054 slices
Permutation processed: 304/1054 slices
Permutation processed: 320/1054 slices
Permutation processed: 336/1054 slices
Permutation processed: 352/1054 slices
Permutation processed: 368/1054 slices
Permutation processed: 384/1054 slices
Permutation processed: 400/1054 slices
Permutation processed: 416/1054

###MERGING - SHAP + PERMUTATION

In [16]:
# Merge SHAP and permutation results
combined_df = saved_shap.merge(
    permutation_scores_df[
        ["SUB_ID", "SLICE_PATH", "PERMUTATION_SCORE"]
    ],
    on=["SUB_ID", "SLICE_PATH"],
    how="inner"
)

print("Combined slices:", len(combined_df))
print(combined_df.head())

print("\nSHAP range:")
print(
    combined_df["SHAP_SCORE"].min(),
    combined_df["SHAP_SCORE"].max()
)

print("\nPermutation range:")
print(
    combined_df["PERMUTATION_SCORE"].min(),
    combined_df["PERMUTATION_SCORE"].max()
)

Combined slices: 1054
   SUB_ID                                         SLICE_PATH    LABEL  \
0   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
1   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
2   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
3   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   
4   51487  /content/drive/MyDrive/ASD_XAI_Project/slices/...  Control   

   SHAP_SCORE  PERMUTATION_SCORE  
0    0.000295           0.004262  
1    0.000283           0.058933  
2    0.000245           0.036996  
3    0.000214           0.037339  
4    0.000264           0.024345  

SHAP range:
9.132681976813558e-05 0.0003567693404611

Permutation range:
0.0 0.6963183879852295


In [17]:
# Min-Max normalization

shap_min = combined_df["SHAP_SCORE"].min()
shap_max = combined_df["SHAP_SCORE"].max()

perm_min = combined_df["PERMUTATION_SCORE"].min()
perm_max = combined_df["PERMUTATION_SCORE"].max()


combined_df["SHAP_NORM"] = (
    (combined_df["SHAP_SCORE"] - shap_min)
    / (shap_max - shap_min)
)

combined_df["PERMUTATION_NORM"] = (
    (combined_df["PERMUTATION_SCORE"] - perm_min)
    / (perm_max - perm_min)
)


# Equal-weight XAI score
ALPHA = 0.5
BETA = 0.5

combined_df["COMBINED_SCORE"] = (
    ALPHA * combined_df["SHAP_NORM"]
    + BETA * combined_df["PERMUTATION_NORM"]
)


print("SHAP normalized range:")
print(
    combined_df["SHAP_NORM"].min(),
    combined_df["SHAP_NORM"].max()
)

print("\nPermutation normalized range:")
print(
    combined_df["PERMUTATION_NORM"].min(),
    combined_df["PERMUTATION_NORM"].max()
)

print("\nTop 10 combined XAI-ranked slices:")

print(
    combined_df
    .sort_values("COMBINED_SCORE", ascending=False)
    [["SUB_ID", "LABEL", "SHAP_NORM",
      "PERMUTATION_NORM", "COMBINED_SCORE"]]
    .head(10)
)

SHAP normalized range:
0.0 1.0

Permutation normalized range:
0.0 1.0

Top 10 combined XAI-ranked slices:
     SUB_ID    LABEL  SHAP_NORM  PERMUTATION_NORM  COMBINED_SCORE
832   51476  Control   1.000000          0.521220        0.760610
914   51466      ASD   0.588689          0.928687        0.758688
326   51460      ASD   0.877069          0.594684        0.735876
491   51463      ASD   0.764150          0.696449        0.730299
829   51476  Control   0.787125          0.660102        0.723613
913   51466      ASD   0.501323          0.926544        0.713933
492   51463      ASD   0.716083          0.710599        0.713341
909   51466      ASD   0.425942          0.986875        0.706409
664   51475  Control   0.713942          0.697537        0.705740
670   51475  Control   0.741271          0.661771        0.701521


In [18]:
subject_summary = (
    combined_df
    .groupby(["SUB_ID", "LABEL"])
    .size()
    .reset_index(name="NUM_SLICES")
)

print("Total validation subjects:", subject_summary["SUB_ID"].nunique())

print("\nSlices per subject:")
print(subject_summary)

print("\nSlice count statistics:")
print(subject_summary["NUM_SLICES"].describe())

Total validation subjects: 6

Slices per subject:
   SUB_ID    LABEL  NUM_SLICES
0   51460      ASD         176
1   51463      ASD         176
2   51466      ASD         176
3   51475  Control         176
4   51476  Control         174
5   51487  Control         176

Slice count statistics:
count      6.000000
mean     175.666667
std        0.816497
min      174.000000
25%      176.000000
50%      176.000000
75%      176.000000
max      176.000000
Name: NUM_SLICES, dtype: float64


In [19]:
TOP_K = 20

# Rank slices independently inside each subject
ranked_df = combined_df.copy()

ranked_df["SLICE_RANK"] = (
    ranked_df
    .groupby("SUB_ID")["COMBINED_SCORE"]
    .rank(method="first", ascending=False)
)

# Select Top-K slices from every MRI subject
selected_slices_df = (
    ranked_df[
        ranked_df["SLICE_RANK"] <= TOP_K
    ]
    .sort_values(
        ["SUB_ID", "SLICE_RANK"]
    )
    .reset_index(drop=True)
)

print("Top-K:", TOP_K)
print("Total selected slices:", len(selected_slices_df))

print("\nSelected slices per subject:")
print(
    selected_slices_df
    .groupby(["SUB_ID", "LABEL"])
    .size()
)

print("\nTop 5 slices of each subject:")

print(
    selected_slices_df
    .groupby("SUB_ID")
    .head(5)[
        [
            "SUB_ID",
            "LABEL",
            "SHAP_NORM",
            "PERMUTATION_NORM",
            "COMBINED_SCORE",
            "SLICE_RANK"
        ]
    ]
)

Top-K: 20
Total selected slices: 120

Selected slices per subject:
SUB_ID  LABEL  
51460   ASD        20
51463   ASD        20
51466   ASD        20
51475   Control    20
51476   Control    20
51487   Control    20
dtype: int64

Top 5 slices of each subject:
     SUB_ID    LABEL  SHAP_NORM  PERMUTATION_NORM  COMBINED_SCORE  SLICE_RANK
0     51460      ASD   0.877069          0.594684        0.735876         1.0
1     51460      ASD   0.603615          0.619110        0.611362         2.0
2     51460      ASD   0.794251          0.373334        0.583793         3.0
3     51460      ASD   0.540841          0.593711        0.567276         4.0
4     51460      ASD   0.832732          0.263515        0.548124         5.0
20    51463      ASD   0.764150          0.696449        0.730299         1.0
21    51463      ASD   0.716083          0.710599        0.713341         2.0
22    51463      ASD   0.634058          0.689586        0.661822         3.0
23    51463      ASD   0.691155        

In [20]:
RANKING_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/results/"
    "combined_xai_slice_ranking.csv"
)

ranked_df.to_csv(RANKING_OUTPUT, index=False)

SELECTED_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/results/"
    "top20_xai_selected_slices.csv"
)

selected_slices_df.to_csv(SELECTED_OUTPUT, index=False)

print("Saved combined ranking:", RANKING_OUTPUT)
print("Saved Top-20 selection:", SELECTED_OUTPUT)

print("\nRanking rows:", len(ranked_df))
print("Selected rows:", len(selected_slices_df))

Saved combined ranking: /content/drive/MyDrive/ASD_XAI_Project/results/combined_xai_slice_ranking.csv
Saved Top-20 selection: /content/drive/MyDrive/ASD_XAI_Project/results/top20_xai_selected_slices.csv

Ranking rows: 1054
Selected rows: 120


In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

def evaluate_subject_level(df, model):

    results = []

    for sub_id, group in df.groupby("SUB_ID"):

        images = np.stack([
            load_image(path).numpy()
            for path in group["SLICE_PATH"]
        ])

        predictions = model.predict(
            images,
            batch_size=16,
            verbose=0
        ).reshape(-1)

        # Average slice probabilities → subject probability
        subject_probability = np.mean(predictions)

        predicted_label = (
            "ASD"
            if subject_probability >= 0.5
            else "Control"
        )

        true_label = group["LABEL"].iloc[0]

        results.append({
            "SUB_ID": sub_id,
            "TRUE_LABEL": true_label,
            "PREDICTED_LABEL": predicted_label,
            "PROBABILITY": subject_probability,
            "NUM_SLICES": len(group)
        })

    results_df = pd.DataFrame(results)

    accuracy = accuracy_score(
        results_df["TRUE_LABEL"],
        results_df["PREDICTED_LABEL"]
    )

    return results_df, accuracy

In [22]:
# Baseline: all validation slices

baseline_results, baseline_accuracy = evaluate_subject_level(
    val_slices_df,
    model
)

print("=== BASELINE: ALL SLICES ===")
print(baseline_results)

print(
    "\nSubject-level accuracy:",
    baseline_accuracy
)

=== BASELINE: ALL SLICES ===
   SUB_ID TRUE_LABEL PREDICTED_LABEL  PROBABILITY  NUM_SLICES
0   51460        ASD         Control     0.223681         176
1   51463        ASD         Control     0.329665         176
2   51466        ASD             ASD     0.655260         176
3   51475    Control             ASD     0.606719         176
4   51476    Control             ASD     0.500834         174
5   51487    Control             ASD     0.542210         176

Subject-level accuracy: 0.16666666666666666


In [23]:
# XAI Top-20 selected slices

xai20_results, xai20_accuracy = evaluate_subject_level(
    selected_slices_df,
    model
)

print("=== XAI TOP-20 SLICES ===")
print(xai20_results)

print(
    "\nSubject-level accuracy:",
    xai20_accuracy
)

print("\n=== COMPARISON ===")
print(
    f"All slices accuracy : {baseline_accuracy:.4f}"
)

print(
    f"XAI Top-20 accuracy : {xai20_accuracy:.4f}"
)

print(
    f"Accuracy difference : "
    f"{xai20_accuracy - baseline_accuracy:+.4f}"
)

=== XAI TOP-20 SLICES ===
   SUB_ID TRUE_LABEL PREDICTED_LABEL  PROBABILITY  NUM_SLICES
0   51460        ASD         Control     0.248746          20
1   51463        ASD             ASD     0.550712          20
2   51466        ASD             ASD     0.868578          20
3   51475    Control             ASD     0.738405          20
4   51476    Control             ASD     0.693467          20
5   51487    Control             ASD     0.748069          20

Subject-level accuracy: 0.3333333333333333

=== COMPARISON ===
All slices accuracy : 0.1667
XAI Top-20 accuracy : 0.3333
Accuracy difference : +0.1667


In [24]:
K_VALUES = [5, 10, 15, 20, 30, 40, 50]

k_results = []

for k in K_VALUES:

    current_selected = (
        ranked_df[
            ranked_df["SLICE_RANK"] <= k
        ]
        .copy()
    )

    subject_results, accuracy = evaluate_subject_level(
        current_selected,
        model
    )

    k_results.append({
        "TOP_K": k,
        "TOTAL_SLICES": len(current_selected),
        "ACCURACY": accuracy
    })

    print(
        f"K={k:2d} | "
        f"Slices={len(current_selected):3d} | "
        f"Accuracy={accuracy:.4f}"
    )


k_results_df = pd.DataFrame(k_results)

print("\n=== TOP-K COMPARISON ===")
print(k_results_df)

best_row = k_results_df.loc[
    k_results_df["ACCURACY"].idxmax()
]

print("\nBest validation setting:")
print("K =", int(best_row["TOP_K"]))
print("Accuracy =", best_row["ACCURACY"])
print("Total slices =", int(best_row["TOTAL_SLICES"]))

K= 5 | Slices= 30 | Accuracy=0.3333
K=10 | Slices= 60 | Accuracy=0.3333
K=15 | Slices= 90 | Accuracy=0.3333
K=20 | Slices=120 | Accuracy=0.3333
K=30 | Slices=180 | Accuracy=0.1667
K=40 | Slices=240 | Accuracy=0.1667
K=50 | Slices=300 | Accuracy=0.1667

=== TOP-K COMPARISON ===
   TOP_K  TOTAL_SLICES  ACCURACY
0      5            30  0.333333
1     10            60  0.333333
2     15            90  0.333333
3     20           120  0.333333
4     30           180  0.166667
5     40           240  0.166667
6     50           300  0.166667

Best validation setting:
K = 5
Accuracy = 0.3333333333333333
Total slices = 30


In [26]:
K_RESULTS_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/results/"
    "topk_validation_comparison.csv"
)

k_results_df.to_csv(
    K_RESULTS_OUTPUT,
    index=False
)

# Save Top-5 candidate
top5_slices_df = (
    ranked_df[
        ranked_df["SLICE_RANK"] <= 5
    ]
    .sort_values(["SUB_ID", "SLICE_RANK"])
    .reset_index(drop=True)
)

TOP5_OUTPUT = (
    "/content/drive/MyDrive/ASD_XAI_Project/results/"
    "top5_xai_selected_slices.csv"
)

top5_slices_df.to_csv(
    TOP5_OUTPUT,
    index=False
)

print("Top-K experiment saved!")
print("Top-5 selected slices saved!")
print("Top-5 rows:", len(top5_slices_df))

Top-K experiment saved!
Top-5 selected slices saved!
Top-5 rows: 30
